# ⚡ Gurukuladesam Ultra-Batch Pipeline (NVIDIA L4 24GB GPU)
## Fully Automated Autonomous Multi-Chapter Parallel Processing

- **Architecture**: NVIDIA Ada Lovelace L4 (24GB VRAM) with Dual 8th Gen NVENC.
- **Throughput**: 4 chapters concurrently × 8 parallel camera moves = **32 active NVENC worker threads**.
- **Saturates**: ~12–16 GB of GPU VRAM and all CPU cores.
- **Autonomous**: Automatically syncs audio, all 37 chapter scene images, and builds master films.


In [ ]:
# Cell 1: Environment & GPU Verification
!nvidia-smi
!ffmpeg -f lavfi -i testsrc=duration=2:size=1280x720:rate=24 -c:v h264_nvenc -preset p4 -y /tmp/test_nvenc.mp4 2>&1 | tail -n 6
!ls -lh /tmp/test_nvenc.mp4


In [ ]:
# Cell 2: Clone repository & setup dependencies
import os
if not os.path.exists('/content/Gurukuladesam'):
    !git clone -b main https://github.com/shishyan/GuruKulaDesam.git /content/Gurukuladesam
else:
    !cd /content/Gurukuladesam && git pull origin main

%cd /content/Gurukuladesam
!pip install -q Pillow numpy tqdm requests mutagen


In [ ]:
# Cell 3: Autonomous Full Asset Sync (Audio + All Chapter Images + Loops)
import urllib.request, zipfile, os, json, base64, requests
from pathlib import Path

tunnel_url = 'https://ed017c1c68a7daaa-49-206-115-211.serveousercontent.com'

# 1. Unpack production scene images
zip_dest = Path('/content/production_scene_images.zip')
if not (Path('/content/Gurukuladesam/production/scene_images/ch004').exists() and len(list(Path('/content/Gurukuladesam/production/scene_images/ch004').glob('*.jpg'))) > 10):
    print('Downloading production_scene_images.zip...')
    with requests.get(f'{tunnel_url}/production_scene_images.zip', stream=True) as r:
        with open(zip_dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=4*1024*1024):
                if chunk: f.write(chunk)
    print('Extracting scene images...')
    with zipfile.ZipFile(zip_dest, 'r') as z:
        z.extractall('/content/Gurukuladesam')
    print('✔ All 37 chapter scene images extracted.')

# 2. Unpack audio tracks
audio_dir = Path('/content/audio')
audio_dir.mkdir(parents=True, exist_ok=True)
audio_zip = Path('/content/all_37_audios.zip')
if len(list(audio_dir.glob('audio_ch*.wav'))) < 30:
    print('Downloading all_37_audios.zip...')
    with requests.get(f'{tunnel_url}/all_37_audios.zip', stream=True) as r:
        with open(audio_zip, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if chunk: f.write(chunk)
    print('Extracting audio tracks...')
    with zipfile.ZipFile(audio_zip, 'r') as z:
        z.extractall('/content/audio')
    print(f'✔ Extracted {len(list(audio_dir.glob("*.wav")))} audio tracks.')

# 3. Upward drizzle loop
drizzle = Path('/content/Gurukuladesam/production/upward_divine_drizzle_loop.mp4')
if not drizzle.exists() or drizzle.stat().st_size == 0:
    r = requests.get(f'{tunnel_url}/upward_divine_drizzle_loop.mp4')
    drizzle.write_bytes(r.content)
    print('✔ Downloaded upward divine drizzle loop.')


In [ ]:
# Cell 4: Launch 4-Chapter Concurrent Multi-Threaded GPU Batch Render
%cd /content/Gurukuladesam
!python tools/batch_render_all_cinematic.py --gpu nvenc --start-chapter 4 --parallel-chapters 4
